# Neural Network xG Model

In [148]:
import pandas as pd
from football_analytics.utils import parsing, supabase, shot_geometry
import football_analytics.visuals.shots as shots
from sklearn.metrics import log_loss
from importlib import reload
import numpy as np
from sklearn.model_selection import train_test_split
import numpy as np
import json
from pathlib import Path
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from datetime import datetime
import joblib
reload(parsing)
reload(supabase)
reload(shot_geometry)
reload(shots)

<module 'football_analytics.visuals.shots' from 'C:\\Users\\david\\Documents\\Git\\football-analytics\\src\\football_analytics\\visuals\\shots.py'>

### Load Supabase Data

In [102]:
table_name = 'shots'
key_column = 'statsbomb_event_id'

In [103]:
data = supabase.fetch_all_rows_in_batches(table_name=table_name, key_column=key_column, batch_size=2000)

Fetched 2000 rows (total 2000).
Fetched 2000 rows (total 4000).
Fetched 2000 rows (total 6000).
Fetched 2000 rows (total 8000).
Fetched 2000 rows (total 10000).
Fetched 2000 rows (total 12000).
Fetched 2000 rows (total 14000).
Fetched 2000 rows (total 16000).
Fetched 2000 rows (total 18000).
Fetched 2000 rows (total 20000).
Fetched 2000 rows (total 22000).
Fetched 2000 rows (total 24000).
Fetched 2000 rows (total 26000).
Fetched 2000 rows (total 28000).
Fetched 2000 rows (total 30000).
Fetched 2000 rows (total 32000).
Fetched 2000 rows (total 34000).
Fetched 2000 rows (total 36000).
Fetched 2000 rows (total 38000).
Fetched 2000 rows (total 40000).
Fetched 2000 rows (total 42000).
Fetched 2000 rows (total 44000).
Fetched 2000 rows (total 46000).
Fetched 2000 rows (total 48000).
Fetched 2000 rows (total 50000).
Fetched 2000 rows (total 52000).
Fetched 2000 rows (total 54000).
Fetched 2000 rows (total 56000).
Fetched 2000 rows (total 58000).
Fetched 2000 rows (total 60000).
Fetched 2000 r

In [149]:
df_raw = pd.DataFrame(data)

In [150]:
data_players = supabase.fetch_all_rows_in_batches(
    table_name="players",
    key_column="statsbomb_player_id",
    batch_size=2000,
    columns="statsbomb_player_id, position_id, position_name",
)

Fetched 2000 rows (total 2000).
Fetched 2000 rows (total 4000).
Fetched 2000 rows (total 6000).
Fetched 2000 rows (total 8000).
Fetched 1043 rows (total 9043).


In [151]:
df_players = pd.DataFrame(data_players)

In [152]:
# Merging player data with shots data on player ID
df_merged = df_raw.merge(df_players, left_on='shot_taker_id', right_on='statsbomb_player_id', how='left')

In [153]:
df_merged.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 88023 entries, 0 to 88022
Data columns (total 33 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   created_at                  88023 non-null  object 
 1   statsbomb_event_id          88023 non-null  object 
 2   x1                          88023 non-null  float64
 3   y1                          88023 non-null  float64
 4   distance_to_goal            88023 non-null  float64
 5   angle_to_goal_deg           88023 non-null  float64
 6   keeper_distance             88023 non-null  float64
 7   min_defender_distance       88015 non-null  float64
 8   avg_defender_distance       88015 non-null  float64
 9   num_def_in_shot_triangle    88023 non-null  int64  
 10  num_teammates_in_box        88023 non-null  int64  
 11  shot_to_min_def_ratio       88015 non-null  float64
 12  shot_type                   88023 non-null  object 
 13  body_part                   880

## Prepare Data

In [154]:
df = df_merged.dropna().copy()

In [155]:
df['goal_outcome'] = df['outcome'].apply(lambda x: 1 if x=='Goal' else 0)
df['is_with_feet'] = df['body_part'].apply(lambda x: 1 if (x=='Right Foot' or x=='Left Foot') else 0)
df['is_penalty'] = df['shot_type'].apply(lambda x: 1 if x=='Penalty' else 0)

In [156]:
df['is_defender'] = df['position_id'].apply(lambda x: 1 if x in [1,2,3,4,5,6,7,8] else 0)
df['is_midfielder'] = df['position_id'].apply(lambda x: 1 if x in [9,10,11,12,13,14,15,16,17,18,19,20] else 0)
df['is_forward'] = df['position_id'].apply(lambda x: 1 if x in [21,22,23,24,25] else 0)

In [157]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 87075 entries, 0 to 88022
Data columns (total 39 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   created_at                  87075 non-null  object 
 1   statsbomb_event_id          87075 non-null  object 
 2   x1                          87075 non-null  float64
 3   y1                          87075 non-null  float64
 4   distance_to_goal            87075 non-null  float64
 5   angle_to_goal_deg           87075 non-null  float64
 6   keeper_distance             87075 non-null  float64
 7   min_defender_distance       87075 non-null  float64
 8   avg_defender_distance       87075 non-null  float64
 9   num_def_in_shot_triangle    87075 non-null  int64  
 10  num_teammates_in_box        87075 non-null  int64  
 11  shot_to_min_def_ratio       87075 non-null  float64
 12  shot_type                   87075 non-null  object 
 13  body_part                   87075 no

In [158]:
feature_columns = [
    'distance_to_goal',
    'angle_to_goal_deg',
    'keeper_distance',
    'min_defender_distance',
    'num_teammates_in_box',
    'is_penalty',
    'num_def_in_shot_triangle',
    'frac_goal_uncovered',
    'keeper_is_in_shot_triangle',
    'is_with_feet',
    'under_pressure',
    'is_defender',
    'is_midfielder',
    'is_forward'
    ]


In [159]:
X = df[feature_columns].apply(pd.to_numeric, errors="coerce").to_numpy(dtype=float)
y = np.array(df['goal_outcome'])

#### Check for NaN and Inf in Input / Output

In [160]:
if np.isnan(y).sum() > 0:
    raise ValueError("NaN values found in target variable y")

In [161]:
if np.isnan(X).sum() > 0:
    raise ValueError("NaN values found in feature matrix X")

## Define Loader, Model and Optimizer

In [162]:
scaler = StandardScaler().fit(X)
X = scaler.transform(X)

In [163]:
class ShotDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32).unsqueeze(1)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [164]:
dataset = ShotDataset(X, y)
loader = DataLoader(dataset, batch_size=64, shuffle=True)

In [165]:
import torch.nn as nn

class XGModel(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 8),
            nn.ReLU(),
            nn.Linear(8, 1)  # output = logit
        )

    def forward(self, x):
        return self.net(x)


In [171]:
model = XGModel(input_dim=X.shape[1])

In [172]:
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)


In [173]:
np.isnan(X).sum(), np.isinf(X).sum()

(np.int64(0), np.int64(0))

In [174]:
num_epochs = 30

for epoch in range(num_epochs):
    model.train()
    epoch_loss = 0.0

    for X_batch, y_batch in loader:
        optimizer.zero_grad()

        logits = model(X_batch)
        loss = criterion(logits, y_batch)

        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

    print(f"Epoch {epoch+1:02d} | Loss: {epoch_loss / len(loader):.4f}")


Epoch 01 | Loss: 0.2851
Epoch 02 | Loss: 0.2704
Epoch 03 | Loss: 0.2690
Epoch 04 | Loss: 0.2682
Epoch 05 | Loss: 0.2678
Epoch 06 | Loss: 0.2674
Epoch 07 | Loss: 0.2669
Epoch 08 | Loss: 0.2667
Epoch 09 | Loss: 0.2668
Epoch 10 | Loss: 0.2664
Epoch 11 | Loss: 0.2662
Epoch 12 | Loss: 0.2661
Epoch 13 | Loss: 0.2658
Epoch 14 | Loss: 0.2656
Epoch 15 | Loss: 0.2655
Epoch 16 | Loss: 0.2653
Epoch 17 | Loss: 0.2651
Epoch 18 | Loss: 0.2649
Epoch 19 | Loss: 0.2649
Epoch 20 | Loss: 0.2646
Epoch 21 | Loss: 0.2646
Epoch 22 | Loss: 0.2643
Epoch 23 | Loss: 0.2641
Epoch 24 | Loss: 0.2642
Epoch 25 | Loss: 0.2640
Epoch 26 | Loss: 0.2638
Epoch 27 | Loss: 0.2638
Epoch 28 | Loss: 0.2636
Epoch 29 | Loss: 0.2633
Epoch 30 | Loss: 0.2635


In [175]:
def xg_log_loss(y_true, y_pred):
    """
    Computes binary cross-entropy (log loss) for xG predictions.
    """
    return log_loss(y_true, y_pred)

In [176]:
model.eval()
with torch.no_grad():
    logits = model(torch.tensor(X, dtype=torch.float32))
    xg_probs = torch.sigmoid(logits).numpy().flatten()


In [177]:
xg_log_loss_nn = xg_log_loss(y, xg_probs)
print(f"NN xG log loss: {xg_log_loss_nn:.4f}")

NN xG log loss: 0.2616


## Save Model

In [180]:
from pathlib import Path
date_str = datetime.now().strftime('%Y%m%d_%H%M%S')
model_name = f"nn_xg_model_{date_str}.pth"
torch.save(model.state_dict(), f'../nn_models/{model_name}')
joblib.dump(scaler, f'../nn_models/scaler_{date_str}.save')

def save_feature_columns(columns, output_path):
    payload = {"feature_columns": list(columns)}
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    with output_path.open("w", encoding="utf-8") as handle:
        json.dump(payload, handle, indent=2)


feature_columns_path = f"../nn_models/config_{date_str}.json"
save_feature_columns(feature_columns, feature_columns_path)

In [181]:
df['nn_xg'] = xg_probs

In [182]:
df[['nn_xg', 'statsbomb_xg','statsbomb_event_id']].to_csv('nn_xg_model_predictions.csv', index=False)

## Check Differences

In [183]:
event_id = "000c565b-31f4-431c-b8a2-6072416f502d"
shot = df[df['statsbomb_event_id'] == event_id]

In [184]:
shot['outcome']

19    Goal
Name: outcome, dtype: object

In [185]:
X_temp = (
    shot[feature_columns].apply(pd.to_numeric, errors="coerce").to_numpy(dtype=float)
)

In [186]:
X_tmp

array([[-0.59288172,  0.01531466, -0.50834134,  0.54090907, -0.98235895,
        -0.12539671,  0.21049282,  0.30602436,  0.21000549,  0.43885742,
         1.79522753,  2.19291418, -0.89644663, -0.78638679]])

In [187]:
X_tmp = scaler.transform(X_tmp)

In [188]:
X_tmp

array([[-2.26181356, -1.60766871, -1.87841439, -1.06560046, -1.92616528,
        -1.14112104, -0.57018318, -1.16424269, -3.71767797, -1.08604853,
         3.66580945,  5.35285838, -2.70006318, -2.40479096]])

In [189]:
with torch.no_grad():
    logits = model(torch.tensor(X_tmp, dtype=torch.float32))
    xg_tmp = torch.sigmoid(logits).numpy().flatten()

In [190]:
xg_tmp

array([0.01405769], dtype=float32)

In [191]:
shot_dict = parsing.parse_json_field(shot.iloc[0]['full_json'])

In [192]:
fig = shots.plot_shot_details(shot_dict, show=True)